# Survival Analysis for 90-day Mortality in Heart Failure Cohort

Notebook này gồm toàn bộ flow từ đầu đến cuối:
- Load data
- Kiểm tra dữ liệu
- Tạo biến nhóm
- Descriptive statistics
- Kaplan–Meier overall
- Kaplan–Meier theo age / gender / CCI
- Log-rank tests
- Cox univariable
- Cox multivariable
- Kiểm tra proportional hazards assumption
- Xuất bảng kết quả ra CSV


## 1. Cài thư viện (chạy nếu máy chưa có)


In [ ]:
# Nếu chưa cài lifelines thì bỏ comment dòng dưới rồi chạy
# !pip install lifelines


## 2. Import thư viện


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import logrank_test

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)


## 3. Load dữ liệu

Đổi đường dẫn nếu file CSV của bạn nằm chỗ khác.


In [ ]:
file_path = 'cohort_hf_survival_90d.csv'
df = pd.read_csv(file_path)

print('Shape:', df.shape)
print('\nColumns:')
print(df.columns.tolist())
print('\nHead:')
display(df.head())


## 4. Kiểm tra dữ liệu cơ bản


In [ ]:
print('Describe time_to_event_90d and death_90d:')
display(df[['time_to_event_90d', 'death_90d']].describe())

print('Missing values:')
display(df[['time_to_event_90d', 'death_90d']].isnull().sum())

# Nếu muốn lọc các bản ghi thời gian âm
df = df[df['time_to_event_90d'] >= 0].copy()
print('Shape after removing negative time:', df.shape)


## 5. Tạo biến nhóm


In [ ]:
df['age_group'] = df['anchor_age'].apply(lambda x: '<65' if x < 65 else '>=65')
df['cci_group'] = df['cci_without_hf'].apply(lambda x: '<3' if x < 3 else '>=3')
df['gender_male'] = df['gender'].map({'F': 0, 'M': 1})

display(df[['anchor_age', 'age_group', 'cci_without_hf', 'cci_group', 'gender', 'gender_male']].head())


## 6. Descriptive statistics


In [ ]:
print('Total patients:', len(df))
print('Deaths within 90 days:', int(df['death_90d'].sum()))
print('Censored/alive at 90 days:', int((df['death_90d'] == 0).sum()))
print('Mortality rate:', df['death_90d'].mean())

print('\nTime summary among deaths only:')
display(df[df['death_90d'] == 1]['time_to_event_90d'].describe())


In [ ]:
print('Mortality by gender:')
display(df.groupby('gender')['death_90d'].agg(['count', 'sum', 'mean']))

print('Mortality by age_group:')
display(df.groupby('age_group')['death_90d'].agg(['count', 'sum', 'mean']))

print('Mortality by cci_group:')
display(df.groupby('cci_group')['death_90d'].agg(['count', 'sum', 'mean']))


## 7. Kiểm tra các ca tử vong rất sớm (< 1 ngày)


In [ ]:
early_cols = [c for c in ['subject_id', 'hadm_id', 'admittime', 'dischtime', 'death_datetime', 'time_to_event_90d', 'death_90d'] if c in df.columns]
display(df[df['time_to_event_90d'] < 1][early_cols].head(20))


## 8. Kaplan–Meier overall


In [ ]:
kmf = KaplanMeierFitter()

plt.figure(figsize=(8, 6))
kmf.fit(
    durations=df['time_to_event_90d'],
    event_observed=df['death_90d'],
    label='Overall HF cohort'
)
kmf.plot_survival_function()
plt.title('Kaplan-Meier Survival Curve for 90-day Mortality')
plt.xlabel('Days')
plt.ylabel('Survival probability')
plt.grid(True)
plt.show()

overall_survival_day90 = kmf.predict(90)
print('Estimated survival probability at day 90:', overall_survival_day90)


## 9. Hàm vẽ Kaplan–Meier theo nhóm


In [ ]:
def plot_km_by_group(data, group_col, title):
    kmf = KaplanMeierFitter()
    plt.figure(figsize=(8, 6))
    
    for g in data[group_col].dropna().unique():
        mask = data[group_col] == g
        kmf.fit(
            durations=data.loc[mask, 'time_to_event_90d'],
            event_observed=data.loc[mask, 'death_90d'],
            label=f'{group_col}: {g}'
        )
        kmf.plot_survival_function()
    
    plt.title(title)
    plt.xlabel('Days')
    plt.ylabel('Survival probability')
    plt.grid(True)
    plt.show()


## 10. Kaplan–Meier theo age, gender, CCI


In [ ]:
plot_km_by_group(df, 'age_group', 'Kaplan-Meier Survival Curve by Age Group')
plot_km_by_group(df, 'gender', 'Kaplan-Meier Survival Curve by Gender')
plot_km_by_group(df, 'cci_group', 'Kaplan-Meier Survival Curve by CCI Group')


## 11. Log-rank test


In [ ]:
def run_logrank(data, group_col, g1, g2):
    group1 = data[data[group_col] == g1]
    group2 = data[data[group_col] == g2]
    
    result = logrank_test(
        group1['time_to_event_90d'],
        group2['time_to_event_90d'],
        event_observed_A=group1['death_90d'],
        event_observed_B=group2['death_90d']
    )
    
    out = result.summary.copy()
    out['comparison'] = f'{group_col}: {g1} vs {g2}'
    return out

logrank_age = run_logrank(df, 'age_group', '<65', '>=65')
logrank_cci = run_logrank(df, 'cci_group', '<3', '>=3')
logrank_gender = run_logrank(df, 'gender', 'F', 'M')

logrank_results = pd.concat([logrank_age, logrank_cci, logrank_gender], ignore_index=True)
display(logrank_results[['comparison', 'test_statistic', 'p', '-log2(p)']])


## 12. Cox univariable


In [ ]:
def run_univariable_cox(data, var):
    tmp = data[['time_to_event_90d', 'death_90d', var]].dropna().copy()
    cph = CoxPHFitter()
    cph.fit(tmp, duration_col='time_to_event_90d', event_col='death_90d')
    summary = cph.summary.copy()
    summary['variable'] = var
    summary['concordance'] = cph.concordance_index_
    summary['partial_AIC'] = cph.AIC_partial_
    return cph, summary

uni_models = {}
uni_summaries = []

for var in ['anchor_age', 'gender_male', 'cci_without_hf']:
    cph, summary = run_univariable_cox(df, var)
    uni_models[var] = cph
    uni_summaries.append(summary)
    print(f'\n===== {var} =====')
    cph.print_summary()

univariable_results = pd.concat(uni_summaries).reset_index().rename(columns={'covariate': 'term'})
display(univariable_results[['variable', 'term', 'coef', 'exp(coef)', 'se(coef)', 'coef lower 95%', 'coef upper 95%', 'exp(coef) lower 95%', 'exp(coef) upper 95%', 'p', 'concordance', 'partial_AIC']])


## 13. Cox multivariable

Hiện tại dùng 3 biến cơ bản: age, gender, CCI.
Bạn có thể thêm biến khác sau nếu muốn.


In [ ]:
multivariable_cols = ['time_to_event_90d', 'death_90d', 'anchor_age', 'gender_male', 'cci_without_hf']
df_multi = df[multivariable_cols].dropna().copy()

cph_multi = CoxPHFitter()
cph_multi.fit(df_multi, duration_col='time_to_event_90d', event_col='death_90d')
cph_multi.print_summary()

multivariable_results = cph_multi.summary.copy().reset_index().rename(columns={'covariate': 'term'})
display(multivariable_results[['term', 'coef', 'exp(coef)', 'se(coef)', 'coef lower 95%', 'coef upper 95%', 'exp(coef) lower 95%', 'exp(coef) upper 95%', 'p']])

print('Concordance:', cph_multi.concordance_index_)
print('Partial AIC:', cph_multi.AIC_partial_)


## 14. Kiểm tra proportional hazards assumption

Cell này có thể in ra khá nhiều thông tin.


In [ ]:
cph_multi.check_assumptions(df_multi, p_value_threshold=0.05, show_plots=False)


## 15. Tạo bảng kết quả gọn hơn để dùng viết bài


In [ ]:
def tidy_cox_summary(summary_df, model_name):
    out = summary_df.reset_index().rename(columns={'covariate': 'term'}).copy()
    out['model'] = model_name
    cols = ['model', 'term', 'coef', 'exp(coef)', 'exp(coef) lower 95%', 'exp(coef) upper 95%', 'p']
    return out[cols]

tidy_uni = tidy_cox_summary(univariable_results.set_index('term'), 'univariable')
tidy_multi = tidy_cox_summary(multivariable_results.set_index('term'), 'multivariable')

final_cox_results = pd.concat([tidy_uni, tidy_multi], ignore_index=True)
display(final_cox_results)


## 16. Xuất kết quả ra CSV


In [ ]:
logrank_results.to_csv('logrank_results_90d.csv', index=False)
univariable_results.to_csv('cox_univariable_results_90d.csv', index=False)
multivariable_results.to_csv('cox_multivariable_results_90d.csv', index=False)
final_cox_results.to_csv('cox_results_tidy_90d.csv', index=False)

print('Saved files:')
print('- logrank_results_90d.csv')
print('- cox_univariable_results_90d.csv')
print('- cox_multivariable_results_90d.csv')
print('- cox_results_tidy_90d.csv')


## 17. Gợi ý mở rộng sau này

Bạn có thể thêm các biến khác vào multivariable model, ví dụ:
- insurance
- race
- admission_type
- lab values
- SOFA / APS / vitals nếu bạn đã feature engineering xong

Ví dụ:
`multivariable_cols = ['time_to_event_90d', 'death_90d', 'anchor_age', 'gender_male', 'cci_without_hf', 'insurance_xxx', 'sofa_score']`
